In [4]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Set device (Use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Load data
train_df = pd.read_csv("/content/train.csv", engine='python', on_bad_lines='warn')

# 2. Basic Text Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Cleaning text...")
train_df['clean_text'] = train_df['comment_text'].apply(clean_text)

# Split data (We use a smaller sample for faster testing, remove .sample() for full training later)
train_df = train_df.sample(frac=0.5, random_state=42) # Using 50% of data to speed up

labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
X_train, X_val, y_train, y_val = train_test_split(train_df['clean_text'], train_df[labels], test_size=0.2, random_state=42)

Using device: cpu
Cleaning text...


In [5]:
# 3. Build Vocabulary
MAX_WORDS = 15000
MAX_LEN = 100

print("Building vocabulary...")
words = [word for text in X_train for word in text.split()]
word_counts = Counter(words)
vocab = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common(MAX_WORDS))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

# Function to convert text to sequence of integers
def text_to_sequence(text):
    tokens = [vocab.get(word, vocab['<UNK>']) for word in text.split()]
    # Pad or truncate to MAX_LEN
    if len(tokens) < MAX_LEN:
        tokens = tokens + [vocab['<PAD>']] * (MAX_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_LEN]
    return tokens

# 4. Custom PyTorch Dataset
class ToxicDataset(Dataset):
    def __init__(self, texts, targets):
        self.texts = [text_to_sequence(t) for t in texts]
        self.targets = targets.values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(self.texts[idx], dtype=torch.long)
        y = torch.tensor(self.targets[idx], dtype=torch.float)
        return x, y

train_dataset = ToxicDataset(X_train, y_train)
val_dataset = ToxicDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
print("DataLoaders created!")

Building vocabulary...
DataLoaders created!
